In [1]:
from __future__ import annotations

import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "TechnicalNote"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

notes = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="summary",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="body",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="internal_notes",
            data_type=wvc.config.DataType.TEXT,
            skip_vectorization=True,
        ),
        wvc.config.Property(
            name="source_url",
            data_type=wvc.config.DataType.TEXT,
            skip_vectorization=True,
        ),
        wvc.config.Property(
            name="status",
            data_type=wvc.config.DataType.TEXT,
            skip_vectorization=True,
        ),
        wvc.config.Property(
            name="priority",
            data_type=wvc.config.DataType.INT,
            skip_vectorization=True,
        ),
        wvc.config.Property(
            name="published",
            data_type=wvc.config.DataType.BOOL,
            skip_vectorization=True,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: TechnicalNote


In [5]:
technical_notes = [
    {
        "title": "Weaviate Cloud Authentication",
        "summary": "How to connect to Weaviate Cloud using cluster URL and API keys.",
        "body": (
            "Weaviate Cloud connection requires a cluster URL, a Weaviate API key "
            "and provider headers such as the OpenAI API key for vectorization."
        ),
        "internal_notes": "Internal review note: mention billing later, unrelated sales priority.",
        "source_url": "https://internal.example.com/weaviate-cloud-auth",
        "status": "published",
        "priority": 5,
        "published": True,
    },
    {
        "title": "Vector Search Basics",
        "summary": "Semantic search using embeddings and vector similarity.",
        "body": (
            "Vector search converts text into embeddings and finds objects that are "
            "semantically similar to the query."
        ),
        "internal_notes": "Internal note: do not mention Docker here.",
        "source_url": "https://internal.example.com/vector-search",
        "status": "published",
        "priority": 4,
        "published": True,
    },
    {
        "title": "Hybrid Search with Alpha",
        "summary": "How BM25 and vector search are combined in hybrid search.",
        "body": (
            "Hybrid search combines keyword-based BM25 search with semantic vector search. "
            "The alpha parameter controls the balance between both methods."
        ),
        "internal_notes": "Internal QA: compare with old search engine migration.",
        "source_url": "https://internal.example.com/hybrid-search",
        "status": "published",
        "priority": 4,
        "published": True,
    },
    {
        "title": "Focused RAG over Notebooks",
        "summary": "Build RAG using clean notebook chunks and Markdown concept notes.",
        "body": (
            "Focused RAG retrieves relevant chunks from Weaviate and passes them to a "
            "generative model to produce grounded answers."
        ),
        "internal_notes": "Internal note: high business value, sales demo, production roadmap.",
        "source_url": "https://internal.example.com/focused-rag",
        "status": "published",
        "priority": 5,
        "published": True,
    },
    {
        "title": "Multi-Tenancy for SaaS",
        "summary": "Isolate data for multiple customers inside one shared collection schema.",
        "body": (
            "Multi-tenancy allows a SaaS application to keep data isolated for different "
            "customers, companies or organizations."
        ),
        "internal_notes": "Internal note: important for enterprise roadmap.",
        "source_url": "https://internal.example.com/multi-tenancy",
        "status": "draft",
        "priority": 5,
        "published": False,
    },
    {
        "title": "JSONL Snapshot Export",
        "summary": "Export collection objects into JSONL and restore them into another collection.",
        "body": (
            "A JSONL snapshot stores one JSON object per line and is useful for application-level "
            "exports, inspection and migration between collections."
        ),
        "internal_notes": "Internal note: not a full backup system.",
        "source_url": "https://internal.example.com/jsonl-snapshot",
        "status": "published",
        "priority": 3,
        "published": True,
    },
]

In [6]:
result = notes.data.insert_many(technical_notes)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = notes.query.fetch_objects(
    limit=20,
    return_properties=[
        "title",
        "summary",
        "status",
        "priority",
        "published",
        "internal_notes",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Title:", obj.properties["title"])
    print("Summary:", obj.properties["summary"])
    print("Status:", obj.properties["status"])
    print("Priority:", obj.properties["priority"])
    print("Published:", obj.properties["published"])
    print("Internal notes:", obj.properties["internal_notes"])
    print("-" * 80)

UUID: 0000efc0-e84a-4251-a2cb-6764020b28cb
Title: JSONL Snapshot Export
Summary: Export collection objects into JSONL and restore them into another collection.
Status: published
Priority: 3
Published: True
Internal notes: Internal note: not a full backup system.
--------------------------------------------------------------------------------
UUID: 3f50cfda-102a-49ff-970f-d8a817f6ae34
Title: Focused RAG over Notebooks
Summary: Build RAG using clean notebook chunks and Markdown concept notes.
Status: published
Priority: 5
Published: True
Internal notes: Internal note: high business value, sales demo, production roadmap.
--------------------------------------------------------------------------------
UUID: 6083f6b6-396b-4749-bbe7-79c2db463745
Title: Multi-Tenancy for SaaS
Summary: Isolate data for multiple customers inside one shared collection schema.
Status: draft
Priority: 5
Published: False
Internal notes: Internal note: important for enterprise roadmap.
------------------------------

In [8]:
response = notes.query.near_text(
    query="how to connect to cloud with API keys",
    limit=5,
    return_properties=[
        "title",
        "summary",
        "body",
        "status",
        "priority",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Summary:", obj.properties["summary"])
    print("Status:", obj.properties["status"])
    print("Priority:", obj.properties["priority"])
    print("-" * 80)

Distance: 0.44791555404663086
Title: Weaviate Cloud Authentication
Summary: How to connect to Weaviate Cloud using cluster URL and API keys.
Status: published
Priority: 5
--------------------------------------------------------------------------------
Distance: 0.7728842496871948
Title: Multi-Tenancy for SaaS
Summary: Isolate data for multiple customers inside one shared collection schema.
Status: draft
Priority: 5
--------------------------------------------------------------------------------
Distance: 0.837591290473938
Title: Hybrid Search with Alpha
Summary: How BM25 and vector search are combined in hybrid search.
Status: published
Priority: 4
--------------------------------------------------------------------------------
Distance: 0.854846179485321
Title: Vector Search Basics
Summary: Semantic search using embeddings and vector similarity.
Status: published
Priority: 4
--------------------------------------------------------------------------------
Distance: 0.8674343824386597
T

In [9]:
response = notes.query.near_text(
    query="sales roadmap and billing",
    limit=5,
    return_properties=[
        "title",
        "summary",
        "body",
        "internal_notes",
        "status",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Summary:", obj.properties["summary"])
    print("Internal notes:", obj.properties["internal_notes"])
    print("-" * 80)

Distance: 0.7165995836257935
Title: Multi-Tenancy for SaaS
Summary: Isolate data for multiple customers inside one shared collection schema.
Internal notes: Internal note: important for enterprise roadmap.
--------------------------------------------------------------------------------
Distance: 0.7292242646217346
Title: Weaviate Cloud Authentication
Summary: How to connect to Weaviate Cloud using cluster URL and API keys.
Internal notes: Internal review note: mention billing later, unrelated sales priority.
--------------------------------------------------------------------------------
Distance: 0.7364727258682251
Title: Focused RAG over Notebooks
Summary: Build RAG using clean notebook chunks and Markdown concept notes.
Internal notes: Internal note: high business value, sales demo, production roadmap.
--------------------------------------------------------------------------------
Distance: 0.8461254835128784
Title: Hybrid Search with Alpha
Summary: How BM25 and vector search are c

In [10]:
response = notes.query.fetch_objects(
    filters=Filter.by_property("status").equal("published"),
    limit=20,
    return_properties=[
        "title",
        "summary",
        "status",
        "priority",
        "published",
    ],
)

for obj in response.objects:
    print(obj.properties)

{'status': 'published', 'priority': 4, 'summary': 'Semantic search using embeddings and vector similarity.', 'title': 'Vector Search Basics', 'published': True}
{'status': 'published', 'priority': 5, 'summary': 'How to connect to Weaviate Cloud using cluster URL and API keys.', 'title': 'Weaviate Cloud Authentication', 'published': True}
{'status': 'published', 'priority': 4, 'summary': 'How BM25 and vector search are combined in hybrid search.', 'title': 'Hybrid Search with Alpha', 'published': True}
{'status': 'published', 'priority': 5, 'summary': 'Build RAG using clean notebook chunks and Markdown concept notes.', 'title': 'Focused RAG over Notebooks', 'published': True}
{'status': 'published', 'priority': 3, 'summary': 'Export collection objects into JSONL and restore them into another collection.', 'title': 'JSONL Snapshot Export', 'published': True}


In [11]:
response = notes.query.near_text(
    query="RAG with notebooks and generated answers",
    filters=Filter.by_property("status").equal("published"),
    limit=5,
    return_properties=[
        "title",
        "summary",
        "body",
        "status",
        "priority",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Summary:", obj.properties["summary"])
    print("Status:", obj.properties["status"])
    print("Priority:", obj.properties["priority"])
    print("-" * 80)

Distance: 0.33734726905822754
Title: Focused RAG over Notebooks
Summary: Build RAG using clean notebook chunks and Markdown concept notes.
Status: published
Priority: 5
--------------------------------------------------------------------------------
Distance: 0.7621726989746094
Title: Vector Search Basics
Summary: Semantic search using embeddings and vector similarity.
Status: published
Priority: 4
--------------------------------------------------------------------------------
Distance: 0.766258180141449
Title: JSONL Snapshot Export
Summary: Export collection objects into JSONL and restore them into another collection.
Status: published
Priority: 3
--------------------------------------------------------------------------------
Distance: 0.799497663974762
Title: Hybrid Search with Alpha
Summary: How BM25 and vector search are combined in hybrid search.
Status: published
Priority: 4
--------------------------------------------------------------------------------
Distance: 0.82900738716

In [12]:
response = notes.query.fetch_objects(
    filters=Filter.by_property("priority").greater_or_equal(5),
    limit=20,
    return_properties=[
        "title",
        "summary",
        "priority",
        "status",
    ],
)

for obj in response.objects:
    print("Title:", obj.properties["title"])
    print("Priority:", obj.properties["priority"])
    print("Status:", obj.properties["status"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Title: Weaviate Cloud Authentication
Priority: 5
Status: published
Summary: How to connect to Weaviate Cloud using cluster URL and API keys.
--------------------------------------------------------------------------------
Title: Focused RAG over Notebooks
Priority: 5
Status: published
Summary: Build RAG using clean notebook chunks and Markdown concept notes.
--------------------------------------------------------------------------------
Title: Multi-Tenancy for SaaS
Priority: 5
Status: draft
Summary: Isolate data for multiple customers inside one shared collection schema.
--------------------------------------------------------------------------------


In [13]:
CONTROL_COLLECTION_NAME = "TechnicalNoteControl"

if client.collections.exists(CONTROL_COLLECTION_NAME):
    client.collections.delete(CONTROL_COLLECTION_NAME)

control_notes = client.collections.create(
    name=CONTROL_COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="summary",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="body",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="internal_notes",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source_url",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="status",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="priority",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="published",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", CONTROL_COLLECTION_NAME)

Created collection: TechnicalNoteControl


In [14]:
result = control_notes.data.insert_many(technical_notes)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Control import completed.")

Control import completed.


In [15]:
response = notes.query.near_text(
    query="sales roadmap billing enterprise",
    limit=5,
    return_properties=[
        "title",
        "summary",
        "internal_notes",
    ],
    return_metadata=MetadataQuery(distance=True),
)

print("Selective vectorization collection")
print("-" * 80)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Internal notes:", obj.properties["internal_notes"])
    print("-" * 80)

Selective vectorization collection
--------------------------------------------------------------------------------
Distance: 0.7062076926231384
Title: Multi-Tenancy for SaaS
Internal notes: Internal note: important for enterprise roadmap.
--------------------------------------------------------------------------------
Distance: 0.7349703311920166
Title: Weaviate Cloud Authentication
Internal notes: Internal review note: mention billing later, unrelated sales priority.
--------------------------------------------------------------------------------
Distance: 0.7561911344528198
Title: Focused RAG over Notebooks
Internal notes: Internal note: high business value, sales demo, production roadmap.
--------------------------------------------------------------------------------
Distance: 0.8367962837219238
Title: Hybrid Search with Alpha
Internal notes: Internal QA: compare with old search engine migration.
--------------------------------------------------------------------------------
Dist

In [16]:
response = control_notes.query.near_text(
    query="sales roadmap billing enterprise",
    limit=5,
    return_properties=[
        "title",
        "summary",
        "internal_notes",
    ],
    return_metadata=MetadataQuery(distance=True),
)

print("Control collection")
print("-" * 80)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Internal notes:", obj.properties["internal_notes"])
    print("-" * 80)

Control collection
--------------------------------------------------------------------------------
Distance: 0.705299973487854
Title: Multi-Tenancy for SaaS
Internal notes: Internal note: important for enterprise roadmap.
--------------------------------------------------------------------------------
Distance: 0.7439637780189514
Title: Weaviate Cloud Authentication
Internal notes: Internal review note: mention billing later, unrelated sales priority.
--------------------------------------------------------------------------------
Distance: 0.7619848251342773
Title: Focused RAG over Notebooks
Internal notes: Internal note: high business value, sales demo, production roadmap.
--------------------------------------------------------------------------------
Distance: 0.8416316509246826
Title: Hybrid Search with Alpha
Internal notes: Internal QA: compare with old search engine migration.
--------------------------------------------------------------------------------
Distance: 0.869798362

In [17]:
def search_notes(
    query: str,
    *,
    only_published: bool = True,
    min_priority: int | None = None,
    limit: int = 5,
) -> None:
    filters = None

    if only_published:
        filters = Filter.by_property("published").equal(True)

    if min_priority is not None:
        priority_filter = Filter.by_property("priority").greater_or_equal(min_priority)

        if filters is None:
            filters = priority_filter
        else:
            filters = filters & priority_filter

    response = notes.query.near_text(
        query=query,
        filters=filters,
        limit=limit,
        return_properties=[
            "title",
            "summary",
            "body",
            "status",
            "priority",
            "published",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    print("QUERY:", query)
    print("ONLY PUBLISHED:", only_published)
    print("MIN PRIORITY:", min_priority)
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Title:", obj.properties["title"])
        print("Summary:", obj.properties["summary"])
        print("Status:", obj.properties["status"])
        print("Priority:", obj.properties["priority"])
        print("Published:", obj.properties["published"])
        print("-" * 80)

In [18]:
search_notes(
    "how to connect Weaviate Cloud with OpenAI API key",
    min_priority=4,
)

QUERY: how to connect Weaviate Cloud with OpenAI API key
ONLY PUBLISHED: True
MIN PRIORITY: 4
--------------------------------------------------------------------------------
Distance: 0.24533259868621826
Title: Weaviate Cloud Authentication
Summary: How to connect to Weaviate Cloud using cluster URL and API keys.
Status: published
Priority: 5
Published: True
--------------------------------------------------------------------------------
Distance: 0.6945439577102661
Title: Focused RAG over Notebooks
Summary: Build RAG using clean notebook chunks and Markdown concept notes.
Status: published
Priority: 5
Published: True
--------------------------------------------------------------------------------
Distance: 0.7748920917510986
Title: Vector Search Basics
Summary: Semantic search using embeddings and vector similarity.
Status: published
Priority: 4
Published: True
--------------------------------------------------------------------------------
Distance: 0.800393283367157
Title: Hybrid S

In [19]:
search_notes(
    "semantic search using embeddings and vectors",
)

QUERY: semantic search using embeddings and vectors
ONLY PUBLISHED: True
MIN PRIORITY: None
--------------------------------------------------------------------------------
Distance: 0.3262748122215271
Title: Vector Search Basics
Summary: Semantic search using embeddings and vector similarity.
Status: published
Priority: 4
Published: True
--------------------------------------------------------------------------------
Distance: 0.4789299964904785
Title: Hybrid Search with Alpha
Summary: How BM25 and vector search are combined in hybrid search.
Status: published
Priority: 4
Published: True
--------------------------------------------------------------------------------
Distance: 0.7953503727912903
Title: Focused RAG over Notebooks
Summary: Build RAG using clean notebook chunks and Markdown concept notes.
Status: published
Priority: 5
Published: True
--------------------------------------------------------------------------------
Distance: 0.8335338830947876
Title: Weaviate Cloud Authent

In [20]:
search_notes(
    "RAG over notebooks and markdown notes",
    min_priority=5,
)

QUERY: RAG over notebooks and markdown notes
ONLY PUBLISHED: True
MIN PRIORITY: 5
--------------------------------------------------------------------------------
Distance: 0.4153168201446533
Title: Focused RAG over Notebooks
Summary: Build RAG using clean notebook chunks and Markdown concept notes.
Status: published
Priority: 5
Published: True
--------------------------------------------------------------------------------
Distance: 0.83176589012146
Title: Weaviate Cloud Authentication
Summary: How to connect to Weaviate Cloud using cluster URL and API keys.
Status: published
Priority: 5
Published: True
--------------------------------------------------------------------------------


In [21]:
search_notes(
    "SaaS data isolation with tenants",
    only_published=False,
)

QUERY: SaaS data isolation with tenants
ONLY PUBLISHED: False
MIN PRIORITY: None
--------------------------------------------------------------------------------
Distance: 0.3393140435218811
Title: Multi-Tenancy for SaaS
Summary: Isolate data for multiple customers inside one shared collection schema.
Status: draft
Priority: 5
Published: False
--------------------------------------------------------------------------------
Distance: 0.7108181715011597
Title: Weaviate Cloud Authentication
Summary: How to connect to Weaviate Cloud using cluster URL and API keys.
Status: published
Priority: 5
Published: True
--------------------------------------------------------------------------------
Distance: 0.8059272766113281
Title: Focused RAG over Notebooks
Summary: Build RAG using clean notebook chunks and Markdown concept notes.
Status: published
Priority: 5
Published: True
--------------------------------------------------------------------------------
Distance: 0.8160196542739868
Title: JSONL

In [22]:
response = notes.generate.near_text(
    query="what are the most important production Weaviate concepts",
    filters=Filter.by_property("published").equal(True),
    grouped_task=(
        "Answer using only the retrieved technical notes. "
        "Explain the important Weaviate concepts briefly. "
        "Do not use internal notes. "
        "Mention only public-facing title, summary and body information."
    ),
    limit=5,
    return_properties=[
        "title",
        "summary",
        "body",
        "status",
        "priority",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["title"],
        "| priority:",
        obj.properties["priority"],
        "| status:",
        obj.properties["status"],
    )

Here are the important concepts related to Weaviate:

1. **Weaviate Cloud Authentication**
   - **Summary**: How to connect to Weaviate Cloud using cluster URL and API keys.
   - **Body**: Weaviate Cloud connection requires a cluster URL, a Weaviate API key, and provider headers such as the OpenAI API key for vectorization.

2. **Focused RAG over Notebooks**
   - **Summary**: Build RAG using clean notebook chunks and Markdown concept notes.
   - **Body**: Focused RAG retrieves relevant chunks from Weaviate and passes them to a generative model to produce grounded answers.

3. **Vector Search Basics**
   - **Summary**: Semantic search using embeddings and vector similarity.
   - **Body**: Vector search converts text into embeddings and finds objects that are semantically similar to the query.

4. **Hybrid Search with Alpha**
   - **Summary**: How BM25 and vector search are combined in hybrid search.
   - **Body**: Hybrid search combines keyword-based BM25 search with semantic vector sea

In [23]:
client.close()